# NVARC-Style Data Augmentation
Purpose:
- Download or stage NVARC synthetic puzzles
- Convert puzzles into message-format datasets with augmentation
- Follow NVARC-main/SDG/scripts/build_datasets.py logic
Notes:
- This notebook focuses on data augmentation only
- Requires Kaggle API credentials for downloads


## 1. Config and Paths
Set local paths and output directories.


In [ ]:
# Adjust paths here
from dataclasses import dataclass
from pathlib import Path

@dataclass
class Config:
    repo_root: str = '.'
    nvarc_root: str = 'NVARC-main'
    sdg_root: str = 'SDG'
    synthetic_dir: str = 'SDG/synthetic'
    output_dir: str = 'SDG/data/grids_v15'
    kaggle_zip: str = 'nvarc-synthetic-puzzles.zip'

cfg = Config()
print(cfg)
Path(cfg.synthetic_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)


## 2. Download (Optional)
Download NVARC synthetic puzzles from Kaggle.


In [ ]:
# Requires Kaggle API credentials configured in ~/.kaggle/kaggle.json
# !kaggle datasets download -d sorokin/nvarc-synthetic-puzzles
# !unzip -o nvarc-synthetic-puzzles.zip -d NVARC-main/SDG/synthetic

# If you prefer the already-augmented dataset (contains nvarc_training/full),
# download NVARC Augmented Puzzles instead:
# !kaggle datasets download -d sorokin/nvarc-augmented-puzzles
# !unzip -o nvarc-augmented-puzzles.zip -d NVARC-main/SDG/synthetic


## 3. Inspect Staged Files
Check folder structure after extraction.


In [ ]:
from pathlib import Path

syn_root = Path(cfg.synthetic_dir)
for p in sorted(syn_root.glob('*')):
    print(p)


## 4. Build Augmented Message Datasets (NVARC SDG)
Use NVARC-main/SDG/scripts/build_datasets.py to convert synthetic pairs into message-format datasets.


In [ ]:
import sys
from pathlib import Path

# sdg_scripts = Path(cfg.sdg_root) / 'scripts'
sys.path.append(str(Path(cfg.sdg_root))) # sys.path.append(str(sdg_scripts))

from build_datasets import convert_synthetic_to_messages

# Example: build datasets from synthetic pairs
# These paths match build_datasets.py defaults
nvarc_training_mask = str(Path(cfg.sdg_root) / 'synthetic' / 'pairs' / 'nvarc_training' / '*' / '*.json')
nvarc_full_mask = str(Path(cfg.sdg_root) / 'synthetic' / 'pairs' / 'nvarc_full' / '*' / '*.json')

print('training mask:', nvarc_training_mask)
print('full mask:', nvarc_full_mask)

ds_train = convert_synthetic_to_messages(nvarc_training_mask, seed=6, num_samples=24)
ds_train.save_to_disk(str(Path(cfg.output_dir) / 'nvarc_training'))
print(ds_train)

ds_full = convert_synthetic_to_messages(nvarc_full_mask, seed=7, num_samples=32)
ds_full.save_to_disk(str(Path(cfg.output_dir) / 'nvarc_full'))
print(ds_full)


## 5. (Optional) Verify Augmentation Effects
Sample a few message sequences to confirm augmentation behavior.


In [ ]:
# Inspect a few samples
for row in ds_train.select(range(2)):
    print(row['puzzle_name'])
    for msg in row['messages'][:4]:
        print(msg['role'])
        print(msg['content'])
        print('---')
